In [ ]:
import sys
sys.path.append("..")
sys.path.append("../pipeline")

import os
import time
import warnings

warnings.filterwarnings("ignore", message="PySpark does not yet fully support pandas", category=FutureWarning)

from constants import QUERIES_DIR, get_scenario_paths
from pipeline.spark_session import (
    create_spark_session,
    register_tables_in_spark,
    apply_run_options
)
from pipeline.config_loader import options
from pipeline.benchmark import _print_run_configuration

In [ ]:
spark = create_spark_session("Benja-Test", use_all_cores=False, tune_memory_allocation=False)
run_configuration = apply_run_options(spark, options)
_print_run_configuration(run_configuration)

In [ ]:
spark.sql("SET -v").toPandas()

In [ ]:
spark.conf.getAll

In [ ]:
# conf_df = spark.sql("SET -v").toPandas()   # columns: key, value, meaning, Since version
# conf_df.to_excel("spark_conf.xlsx", index=False)

In [ ]:
SCALE_FACTOR = 1
_, parquet_dir, _ = get_scenario_paths(SCALE_FACTOR)
print(f"Parquet dir: {parquet_dir}")

In [ ]:
register_tables_in_spark(
    spark_session=spark,
    parquet_dir=parquet_dir,
    register_as="views",
    analyze=False,
    enable_static_caching=False
)

In [ ]:
# Turn AQE on for this session only (config.toml has it off).
# Set it before creating the DataFrame, since the plan is built from the conf at planning time.
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

In [ ]:
from contextlib import redirect_stdout
from io import StringIO

def plan_text(df, mode="formatted"):
    buffer = StringIO()
    with redirect_stdout(buffer):
        df.explain(mode=mode)
    return buffer.getvalue()

def join_summary(text):
    return {
        name: text.count(name)
        for name in ("SortMergeJoin", "BroadcastHashJoin", "ShuffledHashJoin")
    }

In [ ]:
QUERY_NUMBER = 72

QUERY_FILE_ORIGINAL = os.path.join(
    QUERIES_DIR,
    "original",
    f"queries_{SCALE_FACTOR}GB",
    f"query{QUERY_NUMBER}.sql"
)
QUERY_SQL_ORIGINAL = open(QUERY_FILE_ORIGINAL, "r").read()

QUERY_FILE_CHANGED = os.path.join(
    QUERIES_DIR,
    "changed",
    f"queries_{SCALE_FACTOR}GB",
    f"query{QUERY_NUMBER}.sql"
)
QUERY_SQL_CHANGED = open(QUERY_FILE_CHANGED, "r").read()

In [ ]:
df = spark.sql(QUERY_SQL_ORIGINAL)   # ONE DataFrame, reused below

plan_before = plan_text(df)
print("BEFORE:", "isFinalPlan=true" in plan_before, join_summary(plan_before))

result_original = df.toPandas()      # executes df's own plan (same as collect())

plan_after = plan_text(df)
print("AFTER: ", "isFinalPlan=true" in plan_after, join_summary(plan_after))

In [ ]:
print(plan_after)   # full final plan; look for AQEShuffleRead nodes and the joins that changed

In [ ]:
def print_join_order(sql):
    plan = spark.sql(sql)._jdf.queryExecution().optimizedPlan()

    def children(node):
        cs = node.children()
        return [cs.apply(i) for i in range(cs.size())]

    def tables(node):
        # Identify the tables under a node by the column prefix of each leaf (cs, inv, w, i, ...)
        kids = children(node)
        if not kids:
            return {node.output().apply(0).name().split("_")[0]}
        return set().union(*(tables(k) for k in kids))

    joins = []
    def visit(node):
        for k in children(node):  # children first, so joins come out in execution order
            visit(k)
        if node.nodeName() == "Join":
            joins.append(node)

    visit(plan)
    for n, j in enumerate(joins, 1):
        left, right = children(j)
        cond = j.condition().get().sql() if j.condition().isDefined() else "(no condition)"
        print(f"{n}. {j.joinType()}: {sorted(tables(left))} ⋈ {sorted(tables(right))}")
        print(f"   ON {cond}")
        print()

In [ ]:
N_RUNS = 1

In [ ]:
df = spark.sql(QUERY_SQL_ORIGINAL)
df.explain(mode="formatted")  # physical plan as a numbered tree plus per-node details; easiest to read
# df.explain(mode="cost")       # optimized logical plan with size/row statistics (useful with CBO)
# df.explain(mode="codegen")    # generated Java code

In [ ]:
for run in range(N_RUNS):
    print(f"Run {run + 1} of {N_RUNS}")
    start_time = time.time()
    result_original = spark.sql(QUERY_SQL_ORIGINAL).toPandas()
    end_time = time.time()
    print(f"Time taken for original query: {end_time - start_time:.2f} seconds")

spark._jvm.System.gc()

In [ ]:
for run in range(N_RUNS):
    print(f"Run {run + 1} of {N_RUNS}")
    start_time = time.time()
    result_changed = spark.sql(QUERY_SQL_CHANGED).toPandas()
    end_time = time.time()
    print(f"Time taken for changed query: {end_time - start_time:.2f} seconds")

spark._jvm.System.gc()


In [ ]:
result_original.equals(result_changed)